In [8]:
# Data exploration LLM_ABSA

In [9]:
import pandas as pd
from src.ML_ABSA import ABSA
from src.evaluator.evaluate_absa import evaluate_absa

# Load dataset once
df = pd.read_csv("../data/Yelp Restaurant Reviews.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
# print("\nSample reviews:")
# print(df['Review Text'].head(10))
df

Dataset shape: (19896, 4)
Columns: ['Yelp URL', 'Rating', 'Date', 'Review Text']


,Yelp URL,Rating,Date,Review Text
0,https://www.yelp.com/biz/sidney-dairy-barn-sidney,5,1/22/2022,All I can say is they have very good ice cream...
1,https://www.yelp.com/biz/sidney-dairy-barn-sidney,4,6/26/2022,Nice little local place for ice cream.My favor...
2,https://www.yelp.com/biz/sidney-dairy-barn-sidney,5,8/7/2021,A delicious treat on a hot day! Staff was very...
3,https://www.yelp.com/biz/sidney-dairy-barn-sidney,4,7/28/2016,This was great service and a fun crew! I got t...
4,https://www.yelp.com/biz/sidney-dairy-barn-sidney,5,6/23/2015,This is one of my favorite places to get ice c...
...,...,...,...,...
19891,https://www.yelp.com/biz/la-pasticceria-las-vegas,4,7/17/2021,Had the chocolate cannoli! The filling was ric...
19892,https://www.yelp.com/biz/la-pasticceria-las-vegas,4,10/21/2019,Love apricot croissant! I bought it at 4:00 PM...
19893,https://www.yelp.com/biz/la-pasticceria-las-vegas,4,10/12/2019,Line was about 25 people long. It went fast! T...
19894,https://www.yelp.com/biz/la-pasticceria-las-vegas,5,4/11/2021,Its hard not to order everything when I come h...


In [10]:
# Initialize analyzer
import os
# In Jupyter notebooks, we need to use a path relative to the notebook location
# Use the local checkpoint copy in the notebooks directory to avoid path issues
checkpoint_path = os.path.join("checkpoints", "ATEPC_MULTILINGUAL_CHECKPOINT")
# Still need the project root for the dataset path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
dataset_path = os.path.join(PROJECT_ROOT, "src", "evaluator", "absa_test.json")

absa = ABSA(model_name=checkpoint_path)
evaluate_absa(absa, dataset_path)


# Run analyzer on 5 random reviews
sample_reviews = df['Review Text'].sample(5)
sample_reviews

Loading PyABSA model 'checkpoints\ATEPC_MULTILINGUAL_CHECKPOINT'...
[2025-10-24 17:54:29] (2.4.2) Load aspect extractor from checkpoints\ATEPC_MULTILINGUAL_CHECKPOINT
[2025-10-24 17:54:29] (2.4.2) config: checkpoints\ATEPC_MULTILINGUAL_CHECKPOINT\fast_lcf_atepc.config
[2025-10-24 17:54:29] (2.4.2) state_dict: checkpoints\ATEPC_MULTILINGUAL_CHECKPOINT\fast_lcf_atepc.state_dict
[2025-10-24 17:54:29] (2.4.2) model: None
[2025-10-24 17:54:29] (2.4.2) tokenizer: checkpoints\ATEPC_MULTILINGUAL_CHECKPOINT\fast_lcf_atepc.tokenizer
[2025-10-24 17:54:30] (2.4.2) Set Model Device: cpu
[2025-10-24 17:54:30] (2.4.2) Device Name: Unknown


C:\Users\phili\venvs\recommenderSystemsProject\Lib\site-packages\transformers\convert_slow_tokenizer.py:560: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
C:\Users\phili\venvs\recommenderSystemsProject\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Precision: 0.734
Recall:    0.622
F1-score:  0.674


17669    Awesome selections with very different and uni...
19723    All of the desserts they sell look and taste a...
12452    This place is pure magic. The gentlemen helpin...
10730    Best donuts I've ever had. They sell out of gl...
17925    Wow! I'm so glad that my husband and I stopped...
Name: Review Text, dtype: object

In [11]:
all_results = []
for i, review in enumerate(sample_reviews, 1):
    results = absa.analyze(str(review))
    print(f"\nRandom Review {i}: {review}")
    for r in results:
        print("   ", r)

    all_results.append({
        "review": review,
        "aspects": [r.aspect for r in results],
        "sentiments": [r.sentiment for r in results]
    })


results_df = pd.DataFrame(all_results)
results_df


C:\Users\phili\venvs\recommenderSystemsProject\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Random Review 1: Awesome selections with very different and unique flavored. It's all made from scratch and the taste proves it! I had a fresh espresso with a scoop of ice cream and the combination was astonishing! Their whisky ice cream is also beyond toothsome. It's a quick walk or ride from west 25th, accessible from the 22 and 26 bus routes.
    AspectSentiment(aspect='selections', sentiment='positive', confidence=0.9982, text_span=[1])
    AspectSentiment(aspect='flavored', sentiment='positive', confidence=0.9977, text_span=[7])
    AspectSentiment(aspect='taste', sentiment='positive', confidence=0.9977, text_span=[9])
    AspectSentiment(aspect='espresso', sentiment='positive', confidence=0.9604, text_span=[4])
    AspectSentiment(aspect='ice cream', sentiment='neutral', confidence=0.6352, text_span=[9, 10])
    AspectSentiment(aspect='whisky ice cream', sentiment='positive', confidence=0.9977, text_span=[1, 2, 3])
    AspectSentiment(aspect='walk', sentiment='positive', confide

,review,aspects,sentiments
0,Awesome selections with very different and uni...,"[selections, flavored, taste, espresso, ice cr...","[positive, positive, positive, positive, neutr..."
1,All of the desserts they sell look and taste a...,"[desserts, serve ice cream, cones, flavors, ma...","[positive, positive, positive, positive, posit..."
2,This place is pure magic. The gentlemen helpin...,"[place, gentlemen, chocolate cake, chocolate p...","[positive, positive, positive, positive, posit..."
3,Best donuts I've ever had. They sell out of gl...,"[donuts, glazed, apricot jelly]","[positive, positive, positive]"
4,Wow! I'm so glad that my husband and I stopped...,"[place, brownie, cream sandwiches, brownies, p...","[positive, positive, positive, positive, neutr..."
